# 04 · Encoding & Evolution — Data Outlives Code

**Part 1 · Data-Intensive Applications** · *Prerequisites: 02–03* · *Book mapping: DDIA ch. 4*

Bytes written by one program are read by another — a different process, a different language,
or (the case people forget) **a different version of the same program, years later**. Systems
are upgraded by **rolling upgrade**: new code deploys to a few nodes at a time, so old and new
code *always* coexist, exchanging data in both directions. Encoding is therefore not a detail;
it is a compatibility contract.

Two directions of that contract, used constantly from here on:

- **Backward compatibility**: *new* code can read data written by *old* code. (Mandatory —
  your database is full of old data.)
- **Forward compatibility**: *old* code can read data written by *new* code. (Mandatory during
  every rollout — and the one that requires actual forethought.)

**Learning objectives.**

1. Explain why language-native serialization and raw JSON both fail as long-term contracts.
2. Build a Protobuf-style **tag-based binary codec** (varints, field tags, unknown-field
   skipping) and derive its evolution rules.
3. Build an Avro-style codec with **writer/reader schema resolution** and derive *its* rules.
4. Produce a compatibility matrix for a schema's history and read it like a reviewer.
5. Reason about the three dataflow modes — databases, services, message brokers — and
   simulate a rolling upgrade that breaks (and one that doesn't).

In [ ]:
import json
import pickle
import struct

import matplotlib.pyplot as plt
import pandas as pd

from sysdes import viz

viz.use_style()

## 1. The tempting dead ends

**Language-native serialization** (`pickle`, Java `Serializable`, ...) couples your *stored
bytes* to your *class definitions* — a refactor becomes a data-corruption event. It also
executes arbitrary code on load (a security hole), and locks out every other language.

In [ ]:
class UserV1:
    def __init__(self, name):
        self.name = name

frozen = pickle.dumps(UserV1("Ada"))     # bytes now secretly reference class 'UserV1'

del UserV1                                # ...a refactor renames/moves the class...
try:
    pickle.loads(frozen)
    raise AssertionError("should have failed")
except AttributeError as e:
    print(f"pickle after refactor: {type(e).__name__}: {e}")
# The data didn't change. The CODE changed, and the data broke. That coupling
# is why pickle is for caches and scratch files, never for stored/shared data.

**Textual formats** (JSON, CSV, XML) decouple from language — a real virtue, and fine for
many uses — but as *long-term contracts* they have holes: no integer/float distinction (and
consumers like JavaScript corrupt integers above 2⁵³), no binary strings, no schema (every
reader re-guesses types — the CSV column that was a ZIP code until "007A" arrived), and
verbosity (field names repeated in every single record).

In [ ]:
big_id = 2**53 + 1
# A JSON document is just text; any consumer that parses numbers as 64-bit
# floats (JavaScript, many analytics tools) silently corrupts large integers:
as_float = float(big_id)
print(f"id sent: {big_id}   id after float64 round trip: {int(as_float)}")
assert int(as_float) != big_id, "integers above 2^53 don't survive float64 parsers"

The fix for all of it: a **schema** — an explicit, versioned description of the data — plus a
binary encoding driven by that schema. Two great designs exist, distinguished by *how a reader
knows what the bytes mean*: tags in the data (Protobuf/Thrift) vs schemas on both sides (Avro).

## 2. Build 1 — a tag-based binary codec (the Protobuf idea)

Two ingredients:

- **Varints**: integers in 7-bit groups, one continuation bit per byte — small numbers cost
  one byte instead of eight. (Negative numbers first go through **zigzag** mapping
  0,-1,1,-2,... → 0,1,2,3,... so they stay small too.)
- **Field tags**: each field is written as `(tag, wire_type)` header + payload. Readers match
  fields by *tag number*, not name — names live only in the schema file, and unknown tags can
  be *skipped* because the wire type says how long they are. That skipping is forward
  compatibility, mechanized.

In [ ]:
def varint_encode(n: int) -> bytes:
    """Unsigned varint: 7 bits per byte, high bit = 'more bytes follow'."""
    if n < 0:
        raise ValueError("varint is unsigned; zigzag first")
    out = bytearray()
    while True:
        bits = n & 0x7F
        n >>= 7
        out.append(bits | (0x80 if n else 0))   # set continuation bit if more
        if not n:
            return bytes(out)

def varint_decode(buf: bytes, pos: int) -> tuple[int, int]:
    """Returns (value, new_pos)."""
    result = shift = 0
    while True:
        byte = buf[pos]
        pos += 1
        result |= (byte & 0x7F) << shift
        if not byte & 0x80:                     # continuation bit clear: done
            return result, pos
        shift += 7

def zigzag(n: int) -> int:
    return (n << 1) ^ (n >> 63)                 # signed -> small unsigned

def unzigzag(z: int) -> int:
    return (z >> 1) ^ -(z & 1)

# Small numbers are tiny; that's most numbers in most data:
assert len(varint_encode(7)) == 1 and len(varint_encode(300)) == 2
assert unzigzag(zigzag(-3)) == -3 and zigzag(-1) == 1 and zigzag(1) == 2
print("varint + zigzag OK: 7 ->", varint_encode(7).hex(),
      " 300 ->", varint_encode(300).hex())

In [ ]:
# Wire types tell a reader how to SKIP a field it doesn't know:
VARINT, LENGTH_DELIMITED = 0, 2

def tag_encode(record: dict, schema: dict[str, tuple[int, str]]) -> bytes:
    """schema: field name -> (tag number, type in {'int','str'}). Fields absent
    from the record are simply not written (all fields are optional)."""
    out = bytearray()
    for name, (tag, ftype) in schema.items():
        if name not in record:
            continue
        if ftype == "int":
            out += varint_encode((tag << 3) | VARINT)
            out += varint_encode(zigzag(record[name]))
        elif ftype == "str":
            payload = record[name].encode()
            out += varint_encode((tag << 3) | LENGTH_DELIMITED)
            out += varint_encode(len(payload)) + payload
        else:
            raise TypeError(f"unsupported type {ftype}")
    return bytes(out)

def tag_decode(buf: bytes, schema: dict[str, tuple[int, str]]) -> dict:
    """Fields are matched by TAG. Unknown tags are skipped using the wire type
    - this single branch is what makes old readers tolerate new writers."""
    by_tag = {tag: (name, ftype) for name, (tag, ftype) in schema.items()}
    record, pos = {}, 0
    while pos < len(buf):
        header, pos = varint_decode(buf, pos)
        tag, wire = header >> 3, header & 0x7
        if wire == VARINT:
            raw, pos = varint_decode(buf, pos)
            value: object = unzigzag(raw)
        elif wire == LENGTH_DELIMITED:
            length, pos = varint_decode(buf, pos)
            value = buf[pos:pos + length].decode()
            pos += length
        else:
            raise ValueError(f"unknown wire type {wire}")
        if tag in by_tag:
            record[by_tag[tag][0]] = value
        # else: unknown tag - value already consumed, safely skipped.
    return record

SCHEMA_V1 = {"id": (1, "int"), "name": (2, "str")}
record = {"id": 1234, "name": "Margaret Hamilton"}
blob_v1 = tag_encode(record, SCHEMA_V1)
assert tag_decode(blob_v1, SCHEMA_V1) == record
print(f"binary: {len(blob_v1)} B vs JSON: {len(json.dumps(record).encode())} B")

### Evolution with tags: the rules and their reasons

In [ ]:
# The v2 schema ADDS a field, with a NEW tag number:
SCHEMA_V2 = {"id": (1, "int"), "name": (2, "str"), "email": (3, "str")}

# BACKWARD: new (v2) reader, old (v1) data -> new field simply absent.
assert tag_decode(blob_v1, SCHEMA_V2) == record
# FORWARD: old (v1) reader, new (v2) data -> unknown tag 3 skipped.
blob_v2 = tag_encode({**record, "email": "mh@nasa.gov"}, SCHEMA_V2)
assert tag_decode(blob_v2, SCHEMA_V1) == record
print("add-a-field with a fresh tag: compatible in both directions")

# FAILURE FIRST - the forbidden move: REUSING a tag number with a new meaning.
# v3 reuses tag 2 (which old data uses for the string 'name') for an int:
SCHEMA_V3_BAD = {"id": (1, "int"), "login_count": (2, "int")}
misread = tag_decode(blob_v1, SCHEMA_V3_BAD)
print("old bytes decoded under the reused tag:", misread)

# No crash. WORSE than a crash: the old 'name' bytes were silently delivered
# as 'login_count'. Type systems downstream may coerce, log, or store this.
assert misread["login_count"] == "Margaret Hamilton", "a name is now a login count"
assert not isinstance(misread["login_count"], int), "silent type corruption"
# This is why tag numbers are forever: you may retire them, never recycle them.

So the tag-format rulebook is short: **add fields only with fresh tags** (old readers skip
them; new readers treat them as optional); **never change a tag's number or type**; **never
make a new field required** (old data doesn't have it). Every rule is a consequence of the
byte format you just implemented.

## 3. Build 2 — schema resolution (the Avro idea)

Avro goes further: **no tags, no field names in the data at all** — just values,
back-to-back, in schema order. Maximum compactness; but now the bytes are unreadable without
knowing *exactly* which schema wrote them. So Avro makes the schema pairing explicit:

> Every read involves **two schemas**: the **writer's** (how the bytes were produced — stored
> with the data file, or fetched from a registry) and the **reader's** (what the application
> wants). The library *resolves* the two: match fields **by name**; writer-only fields are
> dropped; reader-only fields take their **default** — and if there is no default, the read
> fails.

In [ ]:
class SchemaError(Exception):
    pass

def avro_encode(record: dict, schema: dict) -> bytes:
    """Values only, in schema order. Every field must be present."""
    out = bytearray()
    for f in schema["fields"]:
        value = record[f["name"]]
        if f["type"] == "long":
            out += varint_encode(zigzag(value))
        elif f["type"] == "string":
            payload = value.encode()
            out += varint_encode(len(payload)) + payload
        elif f["type"] == "double":
            out += struct.pack("<d", value)
        else:
            raise TypeError(f["type"])
    return bytes(out)

def avro_decode(buf: bytes, writer_schema: dict) -> dict:
    """Decoding NEEDS the writer schema - the bytes carry no field info at all."""
    record, pos = {}, 0
    for f in writer_schema["fields"]:
        if f["type"] == "long":
            raw, pos = varint_decode(buf, pos)
            record[f["name"]] = unzigzag(raw)
        elif f["type"] == "string":
            length, pos = varint_decode(buf, pos)
            record[f["name"]] = buf[pos:pos + length].decode()
            pos += length
        elif f["type"] == "double":
            record[f["name"]] = struct.unpack_from("<d", buf, pos)[0]
            pos += 8
    return record

def avro_read(buf: bytes, writer_schema: dict, reader_schema: dict) -> dict:
    """THE resolution algorithm: decode with the writer's schema, then project
    onto the reader's expectations - defaults fill the gaps, or we fail loudly."""
    written = avro_decode(buf, writer_schema)
    writer_fields = {f["name"]: f for f in writer_schema["fields"]}
    result = {}
    for f in reader_schema["fields"]:
        if f["name"] in written:
            if writer_fields[f["name"]]["type"] != f["type"]:
                raise SchemaError(f"type of {f['name']} changed")
            result[f["name"]] = written[f["name"]]
        elif "default" in f:
            result[f["name"]] = f["default"]     # reader-only field: default
        else:
            raise SchemaError(f"field {f['name']} missing and has no default")
    return result                                # writer-only fields: dropped

V1 = {"name": "User", "fields": [
    {"name": "id", "type": "long"},
    {"name": "name", "type": "string"}]}

blob = avro_encode({"id": 1234, "name": "Margaret Hamilton"}, V1)
assert avro_read(blob, V1, V1) == {"id": 1234, "name": "Margaret Hamilton"}
print(f"avro-style: {len(blob)} B for the same record "
      f"(tags: {len(blob_v1)} B, JSON: {len(json.dumps(record).encode())} B)")

### The compatibility matrix

A schema's history is a sequence of versions; during any rollout (or when reprocessing old
files), **any writer version may meet any reader version**. So compatibility is a *matrix*,
and this is exactly the check a schema registry automates. Four versions of our schema —
one done right, two done wrong:

In [ ]:
V2 = {"name": "User", "fields": [                 # ADD email WITH default: the right way
    {"name": "id", "type": "long"},
    {"name": "name", "type": "string"},
    {"name": "email", "type": "string", "default": ""}]}

V3 = {"name": "User", "fields": [                 # ADD score WITHOUT default: half broken
    {"name": "id", "type": "long"},
    {"name": "name", "type": "string"},
    {"name": "score", "type": "long"}]}

V4 = {"name": "User", "fields": [                 # REMOVE name (which had no default)
    {"name": "id", "type": "long"}]}

versions = {"v1": V1, "v2": V2, "v3": V3, "v4": V4}
sample = {"id": 1, "name": "x", "email": "e", "score": 9}

matrix = pd.DataFrame(index=[f"writer {v}" for v in versions],
                      columns=[f"reader {v}" for v in versions], dtype=object)
for wname, wschema in versions.items():
    payload = avro_encode({f["name"]: sample[f["name"]] for f in wschema["fields"]},
                          wschema)
    for rname, rschema in versions.items():
        try:
            avro_read(payload, wschema, rschema)
            matrix.loc[f"writer {wname}", f"reader {rname}"] = "ok"
        except SchemaError as e:
            matrix.loc[f"writer {wname}", f"reader {rname}"] = "FAIL"
print(matrix.to_string())

# The pattern, asserted:
assert matrix.loc["writer v1", "reader v2"] == "ok",  "add-with-default is backward compatible"
assert matrix.loc["writer v2", "reader v1"] == "ok",  "add-with-default is forward compatible"
assert matrix.loc["writer v1", "reader v3"] == "FAIL", "add-WITHOUT-default breaks backward"
assert matrix.loc["writer v3", "reader v1"] == "ok",   "...though forward still works"
assert matrix.loc["writer v4", "reader v1"] == "FAIL", "removing a no-default field breaks forward"
assert matrix.loc["writer v1", "reader v4"] == "ok",   "...though backward still works"

The Avro rulebook falls out of the algorithm: **only add or remove fields that have
defaults** — then every version reads every other version. (Compare the tag rulebook:
different mechanism — names + defaults vs tags + skipping — same goal.) One more Avro virtue:
because no tags need hand-assigning, schemas can be *generated* from data or code, which is
why Avro fits data pipelines and schema registries so well.

## 4. Sizes — what the schema buys you

In [ ]:
rec = {"id": 1234, "name": "Margaret Hamilton", "email": "mh@nasa.gov"}
sizes = {
    "JSON": len(json.dumps(rec).encode()),
    "tag binary\n(protobuf-style)": len(tag_encode(rec, SCHEMA_V2)),
    "schema binary\n(avro-style)": len(avro_encode(rec, V2)),
}
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(list(sizes), list(sizes.values()), color=["#9e9e9e", "#1565c0", "#2e7d32"])
ax.set_ylabel("bytes per record")
ax.set_title("Field names repeated per record (JSON) vs tags vs schema-implied")
plt.show()

assert sizes["schema binary\n(avro-style)"] < sizes["tag binary\n(protobuf-style)"] \
       < sizes["JSON"]
print({k.split(chr(10))[0]: v for k, v in sizes.items()})

Multiply that per-record difference by billions of rows and the schema formats pay for
themselves in storage and network alone — before counting the compatibility machinery, which
is the real product.

## 5. Dataflow: the three places bytes travel

- **Through databases** — *sending a message to your future self.* Data outlives code by
  years; backward compatibility is non-negotiable. Subtle trap: old code that reads a row
  written by new code, then writes it back, can silently **drop the fields it didn't know**
  — the update trap. (Real ORMs and protobuf runtimes preserve unknown fields for exactly
  this reason.)
- **Through services** (REST/RPC) — client and server are deployed independently; during any
  rollout, requests and responses cross version boundaries in both directions, so you need
  both compatibilities *at once*.
- **Through message brokers** (nb 11) — producer and consumer don't even know each other;
  the schema (with a registry enforcing the matrix above at publish time) *is* the only
  contract between them.

**Failure first: a rolling upgrade, simulated.** A six-node fleet mid-rollout — three nodes
on the old schema, three on the new — where every node sends its records to every other.
First with the disciplined change (v2: add-with-default), then with the careless one
(v3: add-without-default):

In [ ]:
def rollout(old, new) -> tuple[int, int]:
    """Half the fleet writes `old`, half writes `new`; everyone reads everything.
    Returns (successful reads, failed reads)."""
    fleet = [old, old, old, new, new, new]
    ok = fail = 0
    for writer_schema in fleet:
        payload = avro_encode(
            {f["name"]: sample[f["name"]] for f in writer_schema["fields"]},
            writer_schema)
        for reader_schema in fleet:
            try:
                avro_read(payload, writer_schema, reader_schema)
                ok += 1
            except SchemaError:
                fail += 1
    return ok, fail

ok, fail = rollout(V1, V2)
print(f"rollout v1 -> v2 (add WITH default):    {ok} reads ok, {fail} failed")
assert fail == 0, "a disciplined schema change makes rollouts a non-event"

ok, fail = rollout(V1, V3)
print(f"rollout v1 -> v3 (add WITHOUT default): {ok} reads ok, {fail} failed")
assert fail == 9, "every new-node read of old data fails - mid-rollout outage"

Half your fleet erroring on the other half's data, *during* the deploy, with old data in the
database that will *never* be readable by the new code — that is what "we'll just add a
required field" means. The fix costs one `"default":` entry. This asymmetry — tiny discipline,
huge blast radius — is why schema compatibility checks belong in CI, not in code review
vigilance.

## 6. Exercises

1. **Varint by hand.** Encode 300 as a varint (show the two bytes in binary) and decode
   `0xAC 0x02`. Why does the format put the *low* 7 bits first?
2. **Tag archaeology.** A protobuf field `phone = 4` was deleted last year. A new engineer
   adds `premium = 4` (a bool). Describe two concrete ways this goes wrong, given that
   (a) old data files still exist, and (b) one service was never upgraded.
3. **Design the change.** Using Avro rules, evolve `V2` to also carry a list of roles — in a
   way that is compatible in *both* directions. Then name a change to `V2` that is backward-
   but not forward-compatible, and explain when you would knowingly accept that.
4. **The update trap.** Sketch the exact sequence by which an old service that "reads then
   writes back" a database row destroys a new field — and two different fixes.
5. **Registry rules.** Your schema registry offers modes: BACKWARD (new reader reads previous
   version), FORWARD, FULL, and their -TRANSITIVE variants (checked against *all* prior
   versions, not just the latest). A topic keeps 90 days of messages and consumers sometimes
   replay from the start. Which mode is the minimum safe choice, and why?

### Solutions

**1.** 300 = `100101100₂` → low 7 bits `0101100` with continuation bit → `10101100` (0xAC),
then remaining `10` → `00000010` (0x02). So 300 ↔ `AC 02`, and decoding `AC 02` reverses it.
Low-bits-first lets the decoder accumulate with simple shifts as bytes arrive, and small
numbers self-terminate after one byte.

**2.** (a) Batch jobs re-reading last year's files decode old `phone` strings as wire-type-2
payloads where a varint bool is expected — crash at best, silent garbage at worst (you built
this failure in §2). (b) The never-upgraded service still *writes* tag 4 as phone; new
readers interpret real phone numbers as `premium` flags. Both stem from one act: recycling a
tag. Tags are append-only history, like the log in nb 03.

**3.** Add `{"name": "roles", "type": "array<string>", "default": []}` — add-with-default is
compatible both ways (our toy codec lacks arrays; the rule is what matters). Backward-only
change: adding a field with no default is *not* it (that breaks backward); rather, e.g.,
**removing** a no-default field like `name` (writer v-new → reader v-old fails; reader v-new
handles all old data fine) — acceptable when you control and finish all *readers'* upgrades
before any writer upgrades, i.e., consumers first, producers last.

**4.** Old service: `SELECT` row → materializes it into its v1 struct (unknown `email`
dropped) → user edits one field → `UPDATE` writes the whole struct back → `email` is now
gone for that row. Fixes: preserve unknown fields through the read-modify-write cycle (what
protobuf runtimes do), or write **only the changed columns** (`UPDATE ... SET one_field`)
rather than whole-object writes.

**5.** Replaying consumers mean today's reader must read *every* schema version in the
retention window (and old consumers may read new messages until they're upgraded):
FULL_TRANSITIVE — both directions, against all versions — is the minimum that makes replays
and rollouts both non-events. It simply enforces, at publish time, the matrix you built in §3.

## 7. Takeaways

- Rolling upgrades make version coexistence *permanent*; design every format change for both
  **backward** (new reads old — mandatory forever) and **forward** (old reads new — mandatory
  during every deploy) compatibility.
- Tag formats: compatibility = *fresh tags + optional fields + skip unknown*; tags are
  forever. Schema-resolution formats: compatibility = *defaults on anything you add or
  remove*; every read pairs a writer schema with a reader schema.
- Schemas are documentation that cannot rot, compression (no names/tags in the bytes), and a
  machine-checkable gate for CI — the compatibility matrix is the artifact to automate.
- The three dataflows (database / service / broker) are the same problem at three time
  scales: years, minutes, and "whoever consumes this later".

**Further reading.** DDIA ch. 4; the Avro specification (schema resolution section — it's
short); Google's protobuf "dos and don'ts"; Confluent's schema-registry compatibility docs.

**Next:** [05 · Replication](05-replication.ipynb) — the data now travels between *machines
holding copies of it*, and the copies disagree. The distributed systems part of the course
begins.